# Kaggle: Bổ sung user vào TRAIN trên catalog cố định (cho learning curve và huấn luyện lại)

## Mục tiêu
Thêm **user mới** vào tập train của `balanced_u5_i2_v1` mà **không đổi** bất kỳ thứ gì dùng để đánh giá, để kết quả vẫn so sánh được với số hiện tại (tower + reranker v2: test HR@10/50/100 = 3,518 / 6,962 / 9,521 %).

| Giữ nguyên | Thay đổi |
|---|---|
| Catalog 32.557 item (`items.parquet`) và embedding CLIP (không cần trích xuất lại) | `train.parquet`: thêm user mới (toàn bộ positive của họ là train) |
| `valid.parquet` / `test.parquet` (21.690 user đánh giá và target) | |

## Thiết kế (và vì sao)
- **Nguồn:** Amazon Reviews 2023 / Clothing_Shoes_and_Jewelry (cùng nguồn với dataset hiện tại). Notebook tự tải file thô (rất lớn, hàng GB, mất nhiều phút) hoặc dùng bản đã có trong `/kaggle/input`.
- **Cùng định nghĩa benchmark:** chỉ review **đã xác thực**, khử trùng lặp "review mới nhất mỗi cặp user-item", positive = rating ≥ 4, user cần ≥ 5 positive **trong catalog cố định**.
- **Không rò rỉ:** user đã có trong train/valid/test bị loại khỏi pool; user mới không có valid/test nên không bao giờ là user đánh giá. (Protocol hiện tại là leave-last-out theo từng user, không cắt thời gian toàn cục, nên thêm user không đổi tính chất đó.)
- **Cùng phân phối:** chọn user bổ sung sao cho tỉ lệ nhóm hoạt động (5–9 / 10–19 / ≥20 positive) **bằng đúng tỉ lệ user hiện có**; `extra_rank` xếp sao cho **mọi tiền tố** (K user đầu) đều giữ tỉ lệ đó. Nhờ vậy lấy 25%, 50%, 100% pool đều là các điểm hợp lệ trên learning curve. Số user thực tế bị giới hạn bởi nhóm khan hiếm nhất (in ra rõ ràng).

## Đầu vào cần có (Kaggle)
1. Dataset chứa thư mục `balanced_u5_i2_v1` (`items/train/valid/test.parquet`, ~30 MB) lấy từ `data/processed/balanced_u5_i2_v1`.
2. Internet bật (để tải file review thô), hoặc thêm file `Clothing_Shoes_and_Jewelry.jsonl.gz` vào Input.

## Kết quả (`extra_train_users.zip`)
- `extra_train.parquet` + `extra_manifest.json`: dùng cho **learning curve** (mục 11 của `user_tower_training.ipynb` tự phát hiện file này).
- `balanced_u5_i2_v1_plus/`: dataset đã gộp (train = train cũ + toàn bộ user mới; valid/test/items giữ nguyên). Đặt vào `data/processed/` và chạy **huấn luyện thật** bằng `DATN_DATA_DIR=data/processed/balanced_u5_i2_v1_plus` (cho cả `user_tower_training.ipynb` và `reranker_training.ipynb`).

> **Chỉ làm bước huấn luyện thật sau khi learning curve cho thấy dốc đủ lớn.** Lưu ý: thêm user thay đổi popularity/độ cold của item trong train nên cần huấn luyện lại tower **và** reranker; không dùng lẫn checkpoint cũ với dataset `_plus`.

In [ ]:
%pip install -q 'duckdb>=1.1,<2' orjson pyarrow requests psutil

In [ ]:
import os, sys, gc, gzip, hashlib, io, json, shutil, time, zipfile
from pathlib import Path
import duckdb, requests
import pyarrow as pa, pyarrow.parquet as pq
try:
    import orjson; _loads = orjson.loads
except ImportError:            # chạy local không có orjson
    _loads = json.loads

# ==========================================================================
# CẤU HÌNH
# ==========================================================================
EXTRA_MAX_USERS = int(os.environ.get('DATN_EXTRA_MAX_USERS', 100_000))   # trần số user bổ sung; thực tế bị giới hạn bởi số user đủ điều kiện
MIN_USER_POSITIVES = 5        # cùng định nghĩa benchmark u5: >= 5 positive trong catalog (đã khử trùng lặp)
POSITIVE_RATING = 4.0
ONLY_VERIFIED_PURCHASE = True
SEED = '20260930'
RAW_NAME = 'Clothing_Shoes_and_Jewelry.jsonl.gz'
REVIEW_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/' + RAW_NAME

WORK = Path(os.environ['DATN_EXTEND_OUT']) if os.environ.get('DATN_EXTEND_OUT') else (Path('/kaggle/working/extend_users') if Path('/kaggle/working').exists() else Path('extend_users_out'))
WORK.mkdir(parents=True, exist_ok=True)
OUT_DATASET = WORK / 'balanced_u5_i2_v1_plus'      # dataset phiên bản mới (không ghi đè balanced_u5_i2_v1)
OUT_DATASET.mkdir(parents=True, exist_ok=True)

def find_balanced_dir() -> Path:
    """Thư mục chứa items/train/valid/test.parquet của balanced_u5_i2_v1 (upload thành Kaggle Dataset)."""
    roots = ([Path(os.environ['DATN_BALANCED_DIR'])] if os.environ.get('DATN_BALANCED_DIR') else []) + [Path('/kaggle/input'), Path('data/processed')]
    for root in roots:
        if not root.exists(): continue
        for p in sorted(root.rglob('items.parquet')):
            if all((p.parent / f'{n}.parquet').exists() for n in ('train', 'valid', 'test')):
                return p.parent
    raise FileNotFoundError('Không tìm thấy thư mục balanced_u5_i2_v1 (cần items/train/valid/test.parquet). '
                            'Upload data/processed/balanced_u5_i2_v1 (~30 MB) thành Kaggle Dataset rồi thêm vào Input.')

BAL = find_balanced_dir()
def find_raw() -> Path:
    if os.environ.get('DATN_RAW_GZ'): return Path(os.environ['DATN_RAW_GZ'])
    if Path('/kaggle/input').exists():
        for name in (RAW_NAME, 'reviews_raw.jsonl.gz'):
            hit = next(Path('/kaggle/input').rglob(name), None)
            if hit: return hit
    return WORK / 'reviews_raw.jsonl.gz'
RAW_GZ_PATH = find_raw()
con = duckdb.connect(str(WORK / 'processing.duckdb'))
print(f'Balanced dataset: {BAL}\nRaw reviews: {RAW_GZ_PATH} (tồn tại={RAW_GZ_PATH.exists()})\nOutput: {WORK}')


## 1. Catalog cố định và phân phối user hiện có

In [ ]:
# Catalog cố định + danh sách user hiện có (để LOẠI KHỎI pool bổ sung: user đánh giá không được xuất hiện lần nữa)
catalog = set(pq.read_table(BAL / 'items.parquet', columns=['item_id'])['item_id'].to_pylist())
existing_users = set()
for part in ('train', 'valid', 'test'):
    existing_users.update(pq.read_table(BAL / f'{part}.parquet', columns=['user_id'])['user_id'].to_pylist())
print(f'Catalog cố định: {len(catalog):,} item | user hiện có (train/valid/test): {len(existing_users):,}')

# Phân phối mức độ hoạt động của user hiện có (tổng positive qua 3 split) - để chọn user bổ sung CÙNG phân phối.
def band_of(n): return '05_09' if n <= 9 else ('10_19' if n <= 19 else '20_plus')
existing_deg = con.execute(f"""
    SELECT user_id, count(*) n FROM (
      SELECT user_id, is_positive FROM read_parquet('{(BAL / 'train.parquet').as_posix()}')
      UNION ALL SELECT user_id, is_positive FROM read_parquet('{(BAL / 'valid.parquet').as_posix()}')
      UNION ALL SELECT user_id, is_positive FROM read_parquet('{(BAL / 'test.parquet').as_posix()}')
    ) WHERE is_positive = 1 GROUP BY 1""").fetchall()
band_counts = {'05_09': 0, '10_19': 0, '20_plus': 0}
for _, n in existing_deg: band_counts[band_of(n)] += 1
band_share = {b: c / sum(band_counts.values()) for b, c in band_counts.items()}
print('Phân phối activity của user hiện có:', {b: f'{s*100:.1f}%' for b, s in band_share.items()}, '| trung bình positive/user:', round(sum(n for _, n in existing_deg) / len(existing_deg), 2))


## 2. Quét review thô một lượt (user mới, trong catalog, đã xác thực)

In [ ]:
# Quét file review thô MỘT lượt: giữ mọi review đã xác thực, thuộc catalog cố định, của user CHƯA có trong benchmark.
# (giữ cả rating thấp để khử trùng lặp "review mới nhất mỗi cặp user-item" đúng như balanced_u5_i2_v1)
def ensure_raw_download():
    if RAW_GZ_PATH.exists() and RAW_GZ_PATH.stat().st_size > 0:
        print(f'Đã có file thô: {RAW_GZ_PATH} ({RAW_GZ_PATH.stat().st_size/1024**2:.1f} MiB)'); return
    print(f'Đang tải: {REVIEW_URL}'); tmp = RAW_GZ_PATH.with_suffix('.tmp')
    with requests.get(REVIEW_URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk: f.write(chunk)
    tmp.rename(RAW_GZ_PATH); print(f'Đã tải xong ({RAW_GZ_PATH.stat().st_size/1024**2:.1f} MiB)')

raw_candidate = WORK / 'raw_candidate.parquet'
schema = pa.schema([('user_id', pa.string()), ('item_id', pa.string()), ('rating', pa.float32()), ('timestamp', pa.int64()), ('verified_purchase', pa.bool_())])

def stream_candidates():
    ensure_raw_download()
    writer = pq.ParquetWriter(raw_candidate, schema, compression='zstd'); batch, kept, scanned, t0 = [], 0, 0, time.time()
    try:
        with gzip.GzipFile(str(RAW_GZ_PATH)) as gz:
            for line in io.BufferedReader(gz, buffer_size=4 * 1024 * 1024):
                scanned += 1
                if scanned % 2_000_000 == 0: print(f'  đã quét {scanned:,} dòng, giữ {kept + len(batch):,} ({(time.time()-t0)/60:.1f} phút)', flush=True)
                try: row = _loads(line)
                except Exception: continue
                user_id, item_id = row.get('user_id'), row.get('parent_asin')
                if not user_id or item_id not in catalog or user_id in existing_users: continue
                verified = row.get('verified_purchase') is True
                if ONLY_VERIFIED_PURCHASE and not verified: continue
                rating = float(row.get('rating') or 0)
                if rating < 1.0: continue
                batch.append({'user_id': user_id, 'item_id': item_id, 'rating': rating, 'timestamp': int(row.get('timestamp') or 0), 'verified_purchase': verified})
                if len(batch) >= 100_000:
                    writer.write_table(pa.Table.from_pylist(batch, schema=schema)); kept += len(batch); batch.clear()
        if batch:
            writer.write_table(pa.Table.from_pylist(batch, schema=schema)); kept += len(batch)
    finally:
        writer.close()
    print(f'Quét {scanned:,} dòng; giữ {kept:,} review trong catalog của user mới ({raw_candidate.stat().st_size/1024**2:.1f} MiB)')

stream_candidates()


## 3. Khử trùng lặp, chọn user cùng phân phối, xếp hạng lồng nhau

In [ ]:
# Khử trùng lặp -> positive -> user đủ điều kiện -> chọn CÙNG phân phối activity với user hiện có -> xếp hạng lồng nhau
con.execute(f"""CREATE OR REPLACE TABLE dedup AS SELECT * FROM read_parquet('{raw_candidate.as_posix()}')
                QUALIFY row_number() OVER (PARTITION BY user_id, item_id ORDER BY timestamp DESC) = 1""")
con.execute(f"CREATE OR REPLACE TABLE pos AS SELECT user_id, item_id, rating, timestamp, verified_purchase FROM dedup WHERE rating >= {POSITIVE_RATING}")
eligible = con.execute(f"SELECT user_id, count(*) n FROM pos GROUP BY 1 HAVING count(*) >= {MIN_USER_POSITIVES}").fetchall()
avail = {'05_09': [], '10_19': [], '20_plus': []}
for u, n in eligible: avail[band_of(n)].append(u)
print(f'User mới đủ điều kiện (>= {MIN_USER_POSITIVES} positive trong catalog): {len(eligible):,} | theo band:', {b: len(v) for b, v in avail.items()})
assert eligible, 'Không có user mới đủ điều kiện - kiểm tra file thô / catalog'

def h(u): return int.from_bytes(hashlib.blake2b(f'{SEED}:{u}'.encode(), digest_size=8).digest(), 'big')
# Số user tối đa mà vẫn giữ ĐÚNG tỉ lệ band của user hiện có (band khan hiếm nhất là nút cổ chai).
limit = min(len(avail[b]) / band_share[b] for b in avail if band_share[b] > 0)
total = int(min(EXTRA_MAX_USERS, limit))
quota = {b: min(len(avail[b]), round(total * band_share[b])) for b in avail}
bottleneck = min((b for b in avail if band_share[b] > 0), key=lambda b: len(avail[b]) / band_share[b])
print(f'Chọn {sum(quota.values()):,} user (trần {EXTRA_MAX_USERS:,}; band hạn chế: {bottleneck}) | quota theo band: {quota}')
rows = []
for b, users in avail.items():
    chosen = sorted(users, key=h)[:quota[b]]
    for i, u in enumerate(chosen):
        rows.append((u, b, (i + 0.5) / max(len(chosen), 1)))      # vị trí tương đối trong band -> mọi tiền tố đều đúng tỉ lệ band
rows.sort(key=lambda r: (r[2], r[0]))
con.execute('CREATE OR REPLACE TABLE extra_users(user_id VARCHAR, band VARCHAR, extra_rank INTEGER)')
con.executemany('INSERT INTO extra_users VALUES (?, ?, ?)', [(u, b, i) for i, (u, b, _) in enumerate(rows)])
con.execute("""CREATE OR REPLACE TABLE extra_train AS
    SELECT p.user_id, p.item_id, p.rating, p.timestamp, p.verified_purchase, 1 AS is_positive, e.extra_rank
    FROM pos p JOIN extra_users e USING(user_id) ORDER BY e.extra_rank, p.timestamp""")
n_rows, n_users, n_items = con.execute('SELECT count(*), count(DISTINCT user_id), count(DISTINCT item_id) FROM extra_train').fetchone()
print(f'Extra train: {n_rows:,} tương tác | {n_users:,} user | {n_items:,}/{len(catalog):,} item của catalog | {n_rows/n_users:.2f} positive/user')


## 4. Xuất `extra_train.parquet` và dataset gộp `balanced_u5_i2_v1_plus`

In [ ]:
# Xuất: (1) extra_train.parquet (dùng cho learning curve), (2) dataset đã gộp balanced_u5_i2_v1_plus (dùng để huấn luyện thật)
extra_path = WORK / 'extra_train.parquet'
con.execute(f"COPY extra_train TO '{extra_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")

# Gộp vào train: user bổ sung dùng TOÀN BỘ positive làm train (không có valid/test => không bao giờ là user đánh giá).
for name in ('items', 'valid', 'test'):
    shutil.copy2(BAL / f'{name}.parquet', OUT_DATASET / f'{name}.parquet')
con.execute(f"""COPY (
    SELECT user_id, item_id, rating, timestamp, verified_purchase, is_positive, review_title, review_text, helpful_vote
      FROM read_parquet('{(BAL / 'train.parquet').as_posix()}')
    UNION ALL
    SELECT user_id, item_id, rating, timestamp, verified_purchase, is_positive, '' AS review_title, '' AS review_text, 0::INTEGER AS helpful_vote
      FROM extra_train
) TO '{(OUT_DATASET / 'train.parquet').as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)""")

# Chẩn đoán: thêm user thay đổi đồ thị thế nào (đây là cơ chế giúp HR tăng) và có CÙNG phân phối không
def degree_groups(q):
    return dict(con.execute(f"""SELECT CASE WHEN deg = 0 THEN '0 (cold)' WHEN deg <= 2 THEN '1-2' WHEN deg <= 8 THEN '3-8' ELSE '9+' END g, count(*)
        FROM ({q}) GROUP BY 1""").fetchall())
base_deg = f"""SELECT i.item_id, coalesce(d.deg, 0) deg FROM read_parquet('{(BAL / 'items.parquet').as_posix()}') i
    LEFT JOIN (SELECT item_id, count(*) deg FROM read_parquet('{(BAL / 'train.parquet').as_posix()}') WHERE is_positive = 1 GROUP BY 1) d USING(item_id)"""
plus_deg = f"""SELECT i.item_id, coalesce(d.deg, 0) deg FROM read_parquet('{(OUT_DATASET / 'items.parquet').as_posix()}') i
    LEFT JOIN (SELECT item_id, count(*) deg FROM read_parquet('{(OUT_DATASET / 'train.parquet').as_posix()}') WHERE is_positive = 1 GROUP BY 1) d USING(item_id)"""
before, after = degree_groups(base_deg), degree_groups(plus_deg)
extra_band = dict(con.execute('SELECT band, count(*) FROM extra_users GROUP BY 1').fetchall())
manifest = {
    'source': 'Amazon Reviews 2023 / Clothing_Shoes_and_Jewelry', 'strategy': 'extra TRAIN-ONLY users on the fixed balanced_u5_i2_v1 catalog (eval users/targets unchanged)',
    'seed': SEED, 'min_user_positives': MIN_USER_POSITIVES, 'only_verified': ONLY_VERIFIED_PURCHASE, 'positive_rating': POSITIVE_RATING,
    'catalog_items': len(catalog), 'existing_users': len(existing_users), 'extra_users': n_users, 'extra_interactions': n_rows,
    'extra_items_touched': n_items, 'extra_positives_per_user': n_rows / n_users, 'existing_positives_per_user': sum(n for _, n in existing_deg) / len(existing_deg),
    'band_share_existing': band_share, 'band_share_extra': {b: c / n_users for b, c in extra_band.items()},
    'item_train_degree_before': before, 'item_train_degree_after': after, 'extra_rank_note': 'user_id ordered by extra_rank; any prefix keeps the existing band mix'}
(WORK / 'extra_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Item theo bậc train - trước:', before, '\n                      sau  :', after)
print('Mix activity - hiện có:', {b: f'{s*100:.1f}%' for b, s in band_share.items()}, '| bổ sung:', {b: f'{c/n_users*100:.1f}%' for b, c in extra_band.items()})
con.close()
zip_out = WORK / 'extra_train_users.zip'
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    zf.write(extra_path, 'extra_train.parquet'); zf.write(WORK / 'extra_manifest.json', 'extra_manifest.json')
    for f in sorted(OUT_DATASET.glob('*.parquet')): zf.write(f, f'balanced_u5_i2_v1_plus/{f.name}')
print(f'\nHOÀN THÀNH: {zip_out} ({zip_out.stat().st_size/1024**2:.1f} MiB)')


## 5. Sau khi chạy xong
1. Tải `extra_train_users.zip` từ Output.
2. **Learning curve:** đặt `extra_train.parquet` vào `data/` (hoặc thêm làm Kaggle Input) rồi chạy mục 11 của `user_tower_training.ipynb`.
3. **Huấn luyện thật** (chỉ khi dốc đủ lớn): giải nén `balanced_u5_i2_v1_plus/` vào `data/processed/`, đặt `DATN_DATA_DIR`, chạy lại `user_tower_training.ipynb` rồi `reranker_training.ipynb` (reranker nhớ trỏ `DATN_USER_TOWER_DIR` tới tower mới). Test vẫn là 21.690 user và target cũ nên so được với số hiện tại.